# Part 4: Advanced reinforcement learning

While the fundamentals of reinforcement learning focus on value-based approaches like Q-Learning, SARSA, and Monte Carlo, advanced reinforcement learning explores a richer set of tools for solving more complex decision-making problems. These methods go beyond tabular environments and allow agents to handle larger state spaces, continuous actions, and more sophisticated exploration strategies. Instead of relying solely on value functions, advanced RL introduces policy-based methods, actor-critic approaches, eligibility traces, and model-based planning as key extensions to the foundational ideas.

In addition, advanced reinforcement learning broadens the scope of what agents can learn by considering frameworks such as multi-armed bandits for exploration, hierarchical RL for temporally extended actions, and multi-agent RL for cooperative and competitive interactions. Together, these methods provide the stepping stones between classical RL and deep reinforcement learning, equipping learners with the conceptual and practical foundation to understand both algorithmic innovations and real-world applications.

**Policy based methods**

Value-based methods like Q-Learning and SARSA indirectly define a policy by estimating action-value functions and then acting greedily. However, in many environments — especially with large or continuous action spaces — it can be more effective to learn the policy directly. Policy-based methods do this by parameterizing the policy $\pi(a|s, \theta)$, often as a probability distribution over actions given states, and adjusting the parameters $\theta$ to maximize expected rewards.

The key result is the Policy Gradient Theorem, which shows how to estimate gradients of expected returns with respect to policy parameters, allowing us to perform gradient ascent. The simplest algorithm derived from this is REINFORCE, which uses sampled returns from complete episodes to update policy parameters. While it has high variance, it introduces the important idea of optimizing policies directly rather than relying only on value functions.

Example code:

In [6]:
import gymnasium as gym
import numpy as np

env = gym.make("CartPole-v1")
n_actions = env.action_space.n
n_features = env.observation_space.shape[0]

def softmax(x):
    exp_x = np.exp(x - np.max(x))
    return exp_x / np.sum(exp_x)

def policy(state, theta):
    z = state @ theta
    return softmax(z)

def generate_episode(theta):
    states, actions, rewards = [], [], []

    state, info = env.reset()
    done = False

    while not done:
        probs = policy(state, theta)
        action = np.random.choice(len(probs), p=probs)

        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        states.append(state)
        actions.append(action)
        rewards.append(reward)

        state = next_state

    return states, actions, rewards

def compute_returns(rewards, gamma=0.99):
    G = np.zeros(len(rewards))
    G_t = 0

    for t in reversed(range(len(rewards))):
        G_t = rewards[t] + gamma * G_t
        G[t] = G_t

    return G

theta = np.random.rand(n_features, n_actions) * 0.01
alpha = 0.01
episodes = 1000

for ep in range(episodes):
    states, actions, rewards = generate_episode(theta)
    returns = compute_returns(rewards)

    for t in range(len(states)):
        probs = policy(states[t], theta)
        grad = -probs.copy()
        grad[actions[t]] += 1
        theta += alpha * returns[t] * np.outer(states[t], grad)

    if (ep + 1) % 100 == 0:
        print(f"Episode {ep+1}: Total Reward = {sum(rewards)}")

Episode 100: Total Reward = 154.0
Episode 200: Total Reward = 183.0
Episode 300: Total Reward = 377.0
Episode 400: Total Reward = 182.0
Episode 500: Total Reward = 200.0
Episode 600: Total Reward = 210.0
Episode 700: Total Reward = 215.0
Episode 800: Total Reward = 304.0
Episode 900: Total Reward = 273.0
Episode 1000: Total Reward = 37.0


**Actor-Critic Methods**

Value-based methods like Q-Learning and SARSA learn only value functions, while policy-based methods like REINFORCE learn only policies. Both have strengths and weaknesses: value-based methods are sample efficient but struggle with large or continuous action spaces, while policy-based methods handle those spaces naturally but suffer from high variance in their updates. Actor-Critic methods combine the best of both worlds by using two models: an actor that learns the policy and a critic that evaluates the policy through value function estimation.

The actor decides which actions to take by maintaining a parameterized policy. Meanwhile, the critic estimates the expected return (or advantage) of the current state or state-action pair. The critic’s evaluation serves as a baseline for the actor’s policy updates, which reduces the variance compared to Monte Carlo policy gradients like REINFORCE. This makes learning more stable and efficient, especially in environments with long horizons or sparse rewards.

In practice, the actor updates the policy parameters in the direction suggested by the critic’s feedback, while the critic updates its value function parameters using temporal-difference learning. This interaction between actor and critic allows the agent to continuously improve both its policy and its evaluation of future rewards. Actor-Critic methods are foundational for many modern reinforcement learning algorithms, providing a flexible framework that scales to larger and more complex problems.

In [9]:
import gymnasium as gym
import numpy as np

env = gym.make("CartPole-v1")

n_actions = env.action_space.n
n_features = env.observation_space.shape[0]

# Actor parameters (policy)
theta = np.random.rand(n_features, n_actions) * 0.01
# Critic parameters (state-value function)
w = np.random.rand(n_features) * 0.01

alpha_actor = 0.01
alpha_critic = 0.1
gamma = 0.99

def softmax(x):
    exp_x = np.exp(x - np.max(x))
    return exp_x / np.sum(exp_x)

def policy(state):
    z = state @ theta
    return softmax(z)

def value(state):
    return state @ w

episodes = 500
for ep in range(episodes):
    state, info = env.reset()
    done = False
    total_reward = 0

    while not done:
        probs = policy(state)
        action = np.random.choice(n_actions, p=probs)

        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        total_reward += reward

        # TD error (advantage estimate)
        td_target = reward + gamma * (0 if done else value(next_state))
        td_error = td_target - value(state)

        # Critic update
        w += alpha_critic * td_error * state

        # Actor update
        grad = -probs.copy()
        grad[action] += 1
        theta += alpha_actor * td_error * np.outer(state, grad)

        state = next_state

    if (ep + 1) % 50 == 0:
        print(f"Episode {ep+1}: Total Reward = {total_reward}")

Episode 50: Total Reward = 26.0
Episode 100: Total Reward = 8.0
Episode 150: Total Reward = 10.0
Episode 200: Total Reward = 18.0
Episode 250: Total Reward = 22.0
Episode 300: Total Reward = 10.0
Episode 350: Total Reward = 9.0
Episode 400: Total Reward = 17.0
Episode 450: Total Reward = 9.0
Episode 500: Total Reward = 8.0


**Eligibility Traces**

In reinforcement learning, there is often a trade-off between Monte Carlo methods and Temporal Difference (TD) methods. Monte Carlo uses complete returns from episodes, which can lead to unbiased estimates but with high variance and slow learning. TD methods, on the other hand, update after each step using bootstrapped estimates, which provides faster updates but can introduce bias. Eligibility traces offer a way to combine the strengths of both approaches into a single, more flexible method.

An eligibility trace is essentially a short-term memory of how recently each state (or state-action pair) was visited. When an update occurs, not only is the most recent state updated, but also previous states receive credit in proportion to how recently and frequently they were visited. This allows information to propagate more efficiently through the state space. The influence of past states decays over time, controlled by the parameter $\lambda$, which determines the balance between Monte Carlo (when $\lambda = 1$) and TD(0) (when $\lambda = 0$).

The resulting algorithm, often referred to as TD(λ), creates a continuum between pure Monte Carlo and pure TD methods. In practice, this flexibility often leads to faster and more stable learning, because eligibility traces ensure that credit (or blame) for outcomes is assigned not just to the most recent decision but to a sequence of prior decisions. This makes eligibility traces a powerful tool in environments where delayed rewards are common, helping the agent to “connect the dots” between earlier actions and later outcomes more effectively.



Code Example: SARSA(λ) with Eligibility Traces on CliffWalking

In [13]:
import numpy as np
import gymnasium as gym

env = gym.make("CliffWalking-v1")

n_states = env.observation_space.n
n_actions = env.action_space.n

Q = np.zeros((n_states, n_actions))
E = np.zeros((n_states, n_actions))

alpha = 0.1
gamma = 0.99
lam = 0.9
epsilon = 0.1
episodes = 500

def epsilon_greedy(state, epsilon):
    if np.random.rand() < epsilon:
        return np.random.randint(n_actions)
    return np.argmax(Q[state])

for ep in range(episodes):
    state, _ = env.reset()
    action = epsilon_greedy(state, epsilon)
    E.fill(0)
    total_reward = 0
    done = False

    while not done:
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        next_action = epsilon_greedy(next_state, epsilon)

        td_error = reward + gamma * Q[next_state, next_action] * (not done) - Q[state, action]

        E[state, action] += 1
        Q += alpha * td_error * E
        E *= gamma * lam

        state, action = next_state, next_action
        total_reward += reward

    if (ep + 1) % 50 == 0:
        print(f"Episode {ep+1}: Total Reward = {total_reward}")

Episode 50: Total Reward = -25
Episode 100: Total Reward = -17
Episode 150: Total Reward = -25
Episode 200: Total Reward = -17
Episode 250: Total Reward = -19
Episode 300: Total Reward = -19
Episode 350: Total Reward = -19
Episode 400: Total Reward = -20
Episode 450: Total Reward = -21
Episode 500: Total Reward = -25


**Model-based methods**

In the reinforcement learning frameworks we’ve discussed so far (Monte Carlo, TD, SARSA, Q-Learning, Actor-Critic), the agent learns purely from trial and error with the environment. These are known as model-free methods because they don’t assume any knowledge about how the environment works. While powerful, they can require a lot of data to learn effective policies.

Model-based reinforcement learning (MBRL) takes a different approach: in addition to learning a policy or value function, the agent also tries to learn a model of the environment. This model captures the transition dynamics (how states evolve when actions are taken) and sometimes the reward function. Once a model is learned, the agent can use it to plan ahead, simulating possible future trajectories without needing to interact with the real environment at every step.

One well-known algorithm in this area is Dyna-Q. It combines direct reinforcement learning with planning: the agent interacts with the environment to update Q-values as in Q-Learning, but it also uses its learned model to generate additional "imaginary" experiences that help update Q-values more efficiently. This means the agent gets more out of each real interaction, leading to faster convergence and better sample efficiency.

Code example:



In [18]:
import numpy as np
import gymnasium as gym
from collections import defaultdict

env = gym.make("CliffWalking-v1")

n_states = env.observation_space.n
n_actions = env.action_space.n

Q = np.zeros((n_states, n_actions))
model = defaultdict(tuple)   # maps (state, action) -> (next_state, reward)

alpha = 0.1
gamma = 0.99
epsilon = 0.1
planning_steps = 20
episodes = 300

def epsilon_greedy(state, epsilon):
    if np.random.rand() < epsilon:
        return np.random.randint(n_actions)
    return np.argmax(Q[state])

for ep in range(episodes):
    state, _ = env.reset()
    done = False
    total_reward = 0

    while not done:
        action = epsilon_greedy(state, epsilon)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        # Direct RL update (Q-learning)
        best_next = 0 if done else np.max(Q[next_state])
        Q[state, action] += alpha * (reward + gamma * best_next - Q[state, action])

        # Update model
        model[(state, action)] = (next_state, reward)

        # Planning updates
        model_keys = list(model.keys())
        for _ in range(planning_steps):
            s, a = model_keys[np.random.randint(len(model_keys))]
            s_next, r = model[(s, a)]
            best_next_model = np.max(Q[s_next])
            Q[s, a] += alpha * (r + gamma * best_next_model - Q[s, a])

        state = next_state
        total_reward += reward

    if (ep + 1) % 50 == 0:
        print(f"Episode {ep+1}: Total Reward = {total_reward}")

Episode 50: Total Reward = -13
Episode 100: Total Reward = -122
Episode 150: Total Reward = -15
Episode 200: Total Reward = -14
Episode 250: Total Reward = -15
Episode 300: Total Reward = -19
